# Notebook 10: Sklearn Pipelines

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 3 hours

**Week 4 — Data Fundamentals & Preprocessing Pipelines**

---

## Why does this matter for ML?

In Notebook 09 you saw how data leakage silently destroys model validity. The most common source of leakage — fitting preprocessors on test data — has a simple, structural solution: **sklearn Pipelines**.

Without Pipelines, a typical ML workflow involves 7–10 separate `fit()` and `transform()` calls. Each one is a potential point of failure. One misplaced call and you have leakage, a wrong order, or untransformed data going into your model. In team environments, these bugs multiply.

With Pipelines:
- Preprocessing and modelling are a single, atomic object
- `fit()` always means "learn from training data only"
- `predict()` always applies the learned transformations correctly
- Cross-validation is automatically leakage-free
- Deployment is one `joblib.dump()` call

---

## Real-World Analogy: The Assembly Line

Imagine a car factory before assembly lines existed. Workers had to manually pass each part to the next person, remember the correct order, and ensure nothing was skipped. One mistake meant a car with no brakes.

An assembly line automates the sequence: each *station* does one job (weld the frame, install the engine, attach the wheels), in the correct order, every single time. You can inspect any station independently. You can swap out a station for an upgraded version. The line is the same whether you're assembling 1 car or 10,000.

**sklearn Pipelines are assembly lines for data.** Each step processes the data in a fixed, correct order. You can swap components (try a different scaler, a different model). You can inspect any step. The Pipeline works identically whether you call it on training data, test data, or production data.

---

## What We'll Build

```
Numeric Features:
  amount, hour_of_day, distance_from_home_km, n_prev_transactions
     └── SimpleImputer(strategy='median')
         └── StandardScaler()

Categorical Features:
  merchant_category, card_type
     └── SimpleImputer(strategy='most_frequent')
         └── OneHotEncoder(drop='first', handle_unknown='ignore')

     └── ColumnTransformer (combines both branches)
         └── LogisticRegression(class_weight='balanced')
```


In [ ]:
# ============================================================
# CELL 1: Imports and global settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import joblib
import os
import tempfile

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported.')
print(f'numpy {np.__version__} | pandas {pd.__version__}')

## Step 1: Create a Messy, Realistic Fraud Dataset

Real datasets are never clean. Our fraud dataset will have:
- **~15% missing values** across several columns
- **Mixed types**: numeric and categorical columns
- **Class imbalance**: only ~4% of transactions are fraudulent
- **Varying scales**: `amount` ranges 1–5000, `hour_of_day` ranges 0–23

This is the dataset we'll preprocess — first the hard way (manual), then the right way (Pipeline).


In [ ]:
# ============================================================
# CELL 2: Generate a messy synthetic fraud dataset
# ============================================================
N = 8000        # total transactions
FRAUD_RATE = 0.04  # 4% fraud rate — class imbalance is realistic
MISSING_RATE = 0.15  # 15% of values will be randomly set to NaN

rng = np.random.default_rng(RANDOM_STATE)  # reproducible random generator

is_fraud = rng.binomial(1, FRAUD_RATE, N)

# ---- Numeric features ----
amount = np.where(
    is_fraud == 1,
    rng.normal(400, 250, N).clip(5, 5000),
    rng.normal(90, 70, N).clip(1, 2000)
)
hour_of_day = rng.integers(0, 24, N).astype(float)
distance_from_home_km = np.where(
    is_fraud == 1,
    rng.exponential(180, N).clip(0, 20000),
    rng.exponential(12, N).clip(0, 500)
)
# Number of transactions by this card in the last 24 hours
n_prev_transactions = rng.integers(0, 30, N).astype(float)

# ---- Categorical features ----
merchant_category = rng.choice(
    ['grocery', 'online', 'restaurant', 'gas_station', 'retail', 'travel'],
    N, p=[0.25, 0.20, 0.18, 0.12, 0.15, 0.10]
)
card_type = rng.choice(
    ['visa', 'mastercard', 'amex', 'discover'],
    N, p=[0.40, 0.35, 0.15, 0.10]
)

# ---- Assemble clean DataFrame first ----
df_raw = pd.DataFrame({
    'amount':                  amount.round(2),
    'hour_of_day':             hour_of_day,
    'distance_from_home_km':   distance_from_home_km.round(1),
    'n_prev_transactions':     n_prev_transactions,
    'merchant_category':       merchant_category,
    'card_type':               card_type,
    'is_fraud':                is_fraud
})

# ---- Inject missing values realistically ----
# Missing data is more likely in certain columns (e.g., distance not always recorded)
missing_cols = ['amount', 'distance_from_home_km', 'n_prev_transactions',
                'merchant_category', 'card_type']
for col in missing_cols:
    mask = rng.random(N) < MISSING_RATE  # True for ~15% of rows
    df_raw.loc[mask, col] = np.nan

print(f'Dataset shape: {df_raw.shape}')
print(f'Fraud rate:    {df_raw["is_fraud"].mean():.1%}')
print()
print('Missing value counts:')
print(df_raw.isnull().sum().to_string())
print()
print(df_raw.dtypes)

In [ ]:
# ============================================================
# CELL 3: Visualise the messy dataset
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

numeric_cols = ['amount', 'hour_of_day', 'distance_from_home_km', 'n_prev_transactions']
cat_cols = ['merchant_category', 'card_type']

# ---- Numeric distributions: split by fraud label ----
for i, col in enumerate(numeric_cols):
    ax = axes[i]
    # Drop NaN for plotting only; the Pipeline will handle them
    for label, color, name in [(0, '#2ca02c', 'Legitimate'), (1, '#d62728', 'Fraud')]:
        vals = df_raw[df_raw['is_fraud'] == label][col].dropna()
        ax.hist(vals, bins=40, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(f'{col}\n(15% missing)', fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# ---- Categorical bar charts ----
for i, col in enumerate(cat_cols, start=len(numeric_cols)):
    ax = axes[i]
    # Count fraud rate per category (excluding NaN)
    fraud_rates = (
        df_raw.dropna(subset=[col])
        .groupby(col)['is_fraud'].mean().sort_values(ascending=False)
    )
    fraud_rates.plot(kind='bar', ax=ax, color='#1f77b4', alpha=0.8, edgecolor='white')
    ax.set_title(f'{col}: Fraud Rate by Category', fontsize=10)
    ax.set_ylabel('Fraud Rate')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.axhline(y=FRAUD_RATE, linestyle='--', color='red', linewidth=0.8, label='Overall rate')
    ax.legend(fontsize=8)

plt.suptitle('Fraud Dataset: Feature Distributions & Class Separation\n'
             'This messy data needs imputation + scaling + encoding before modelling',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## The Bad Way: Manual Preprocessing (Do NOT do this)

Before showing you Pipelines, let's see exactly what manual preprocessing looks like — and why it's error-prone.

The code below does what many beginners (and even experienced practitioners) write on their first pass. Count the number of steps. Notice how easy it is to:
1. Forget to call `.transform()` on test data with the object fitted on training data
2. Accidentally call `.fit_transform()` on test data instead of `.transform()`
3. Forget to apply the same encoding logic in a different order than expected
4. Lose track of which column is which after stacking arrays


In [ ]:
# ============================================================
# CELL 4: The MANUAL approach — verbose, fragile, error-prone
# ============================================================

# ---- Define feature columns ----
NUM_COLS = ['amount', 'hour_of_day', 'distance_from_home_km', 'n_prev_transactions']
CAT_COLS = ['merchant_category', 'card_type']
TARGET   = 'is_fraud'

X_all = df_raw[NUM_COLS + CAT_COLS]
y_all = df_raw[TARGET]

# STEP 0: Split the data first (correct — always split before fitting anything)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_STATE, stratify=y_all
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

# STEP 1: Impute numeric columns — fit on train, transform both
num_imputer = SimpleImputer(strategy='median')
X_train_num_imp = num_imputer.fit_transform(X_train[NUM_COLS])  # fit+transform train
X_test_num_imp  = num_imputer.transform(X_test[NUM_COLS])       # transform only test

# STEP 2: Scale numeric columns — fit on train, transform both
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num_imp)   # fit+transform train
X_test_num_scaled  = scaler.transform(X_test_num_imp)        # transform only test

# STEP 3: Impute categorical columns — fit on train, transform both
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat_imp = cat_imputer.fit_transform(X_train[CAT_COLS])  # fit+transform train
X_test_cat_imp  = cat_imputer.transform(X_test[CAT_COLS])       # transform only test

# STEP 4: One-hot encode categorical columns — fit on train, transform both
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_train_cat_enc = ohe.fit_transform(X_train_cat_imp)  # fit+transform train
X_test_cat_enc  = ohe.transform(X_test_cat_imp)        # transform only test

# STEP 5: Concatenate numeric and categorical arrays
import numpy as np
X_train_final = np.hstack([X_train_num_scaled, X_train_cat_enc])
X_test_final  = np.hstack([X_test_num_scaled,  X_test_cat_enc])

# STEP 6: Fit model
manual_model = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
)
manual_model.fit(X_train_final, y_train)

# STEP 7: Evaluate
manual_proba = manual_model.predict_proba(X_test_final)[:, 1]
manual_auc = roc_auc_score(y_test, manual_proba)

print()
print(f'Manual pipeline AUC: {manual_auc:.4f}')
print(f'X_train_final shape: {X_train_final.shape}')
print()
print('That was 7 separate fit/transform operations.')
print('Count the places where a single .fit_transform on test data would cause leakage.')

---
## The Right Way: sklearn Pipelines

### How a Pipeline works internally

A Pipeline is a list of `(name, estimator)` tuples. The last step must be an *estimator* (something with `.fit()` and `.predict()`). All earlier steps must be *transformers* (something with `.fit()`, `.transform()`, and `.fit_transform()`).

**When you call `pipeline.fit(X_train, y_train)`:**
1. Step 1: calls `step1.fit_transform(X_train, y_train)` → produces `X_transformed_1`
2. Step 2: calls `step2.fit_transform(X_transformed_1, y_train)` → produces `X_transformed_2`
3. ...
4. Final step: calls `final_estimator.fit(X_transformed_last, y_train)`

**When you call `pipeline.predict(X_test)`:**
1. Step 1: calls `step1.transform(X_test)` (using parameters learned from training)
2. Step 2: calls `step2.transform(...)` (using parameters learned from training)
3. ...
4. Final step: calls `final_estimator.predict(...)` → returns predictions

**Key guarantee:** `.transform()` is called on test data — never `.fit_transform()`. The test data never teaches the Pipeline anything. This is structurally enforced, not just a convention you can accidentally break.


In [ ]:
# ============================================================
# CELL 5: Build the equivalent Pipeline — compare the clarity
# ============================================================

# ---- Numeric sub-pipeline: impute then scale ----
# These steps run sequentially on numeric columns only
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),      # fill NaN with column median
    ('scaler',  StandardScaler())                       # standardise to mean=0, std=1
])

# ---- Categorical sub-pipeline: impute then encode ----
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # fill NaN with most common value
    ('encoder', OneHotEncoder(drop='first',               # drop first to avoid multicollinearity
                              sparse_output=False,        # return dense array (easier to inspect)
                              handle_unknown='ignore'))   # new categories at inference → all zeros
])

# ---- ColumnTransformer: route columns to the right pipeline ----
# 'remainder=drop' means any column not listed is discarded
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,     NUM_COLS),  # numeric cols → numeric_pipeline
    ('cat', categorical_pipeline, CAT_COLS),  # categorical cols → categorical_pipeline
], remainder='drop')

# ---- Full Pipeline: preprocessor + model ----
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',  # compensates for 4% fraud rate imbalance
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# ---- Train and evaluate ----
# ONE call replaces all 7 manual steps
full_pipeline.fit(X_train, y_train)
pipe_proba = full_pipeline.predict_proba(X_test)[:, 1]
pipe_auc   = roc_auc_score(y_test, pipe_proba)

print('Pipeline definition printed above — 3 objects, 1 fit call.')
print()
print(f'Manual approach AUC: {manual_auc:.4f}')
print(f'Pipeline AUC:        {pipe_auc:.4f}')
print()
# The results should be nearly identical — the Pipeline does the same operations
print('Results should be identical (or differ only by floating-point rounding).')
print(f'Difference: {abs(manual_auc - pipe_auc):.6f}')

In [ ]:
# ============================================================
# CELL 6: Visualise the Pipeline architecture
# ============================================================
# sklearn can render a Pipeline diagram directly in Jupyter
# This is extremely useful for verifying the structure before training

from sklearn import set_config
set_config(display='text')  # 'diagram' shows HTML in Jupyter; 'text' works everywhere

print('Pipeline structure:')
print(full_pipeline)
print()
print('Step names accessible via pipeline.named_steps:')
print(list(full_pipeline.named_steps.keys()))
print()
print('Preprocessor step names:')
print([(name, type(trans).__name__) for name, trans, _ in
       full_pipeline.named_steps['preprocessor'].transformers_])

---
## Pipeline Inspection: Looking Inside After Fitting

After calling `pipeline.fit()`, every step stores its learned parameters. You can inspect them:
- `pipeline.named_steps['step_name']` — access a fitted step by name
- `pipeline[:-1].transform(X)` — get the preprocessed features (all steps except the last)
- `pipeline.named_steps['model'].coef_` — model coefficients

This is important for:
- Debugging (checking that scaling looks right)
- Interpretability (examining model weights)
- Monitoring (checking if production data drifts from training statistics)


In [ ]:
# ============================================================
# CELL 7: Inspect the fitted Pipeline's learned parameters
# ============================================================

# ---- Access the scaler inside the numeric sub-pipeline ----
# Path: full_pipeline → preprocessor → 'num' transformer → 'scaler' step
fitted_scaler = (
    full_pipeline
    .named_steps['preprocessor']    # ColumnTransformer
    .named_transformers_['num']      # numeric Pipeline
    .named_steps['scaler']           # StandardScaler
)

print('StandardScaler learned parameters (from TRAINING data only):')
for col, mean, std in zip(NUM_COLS, fitted_scaler.mean_, fitted_scaler.scale_):
    print(f'  {col:<28}: mean = {mean:>10.3f}  |  std = {std:>10.3f}')

print()

# ---- Access the OneHotEncoder inside the categorical sub-pipeline ----
fitted_ohe = (
    full_pipeline
    .named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['encoder']
)

print('OneHotEncoder learned categories (from TRAINING data only):')
for col, cats in zip(CAT_COLS, fitted_ohe.categories_):
    # drop='first' means the first category is dropped; we show all learned categories
    print(f'  {col}: {list(cats)}')

print()

# ---- Access model coefficients ----
fitted_model = full_pipeline.named_steps['model']
print(f'Model type: {type(fitted_model).__name__}')
print(f'Number of coefficients: {fitted_model.coef_.shape[1]}')
print(f'Intercept: {fitted_model.intercept_[0]:.4f}')
print()

# ---- Get the transformed feature matrix ----
# pipeline[:-1] selects all steps except the final estimator
X_preprocessed = full_pipeline[:-1].transform(X_train)
print(f'Preprocessed training matrix shape: {X_preprocessed.shape}')
print('(4 numeric + OHE-encoded categoricals = multiple columns)')

In [ ]:
# ============================================================
# CELL 8: Visualise scaler learned parameters
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- Left: Scaler means ----
ax = axes[0]
bars = ax.barh(NUM_COLS, fitted_scaler.mean_, color='#1f77b4', alpha=0.8)
ax.set_xlabel('Mean Value (learned from training data)')
ax.set_title('StandardScaler: Learned Means\n'
             'These values are subtracted from every input at inference time', fontsize=10)
# Annotate with exact values
for bar, val in zip(bars, fitted_scaler.mean_):
    ax.text(bar.get_width() + ax.get_xlim()[1]*0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)

# ---- Right: OHE feature output visualised on a few rows ----
ax2 = axes[1]
# Show how numeric preprocessing transforms the distributions
# Plot original vs scaled 'amount'
amount_original = X_train['amount'].dropna().values
# Get the scaled version: index 0 in preprocessed matrix = amount
amount_scaled = X_preprocessed[:, 0]  # first column is scaled 'amount'

ax2.hist(amount_original, bins=50, alpha=0.6, color='#ff7f0e',
         label=f'Original (mean={amount_original.mean():.0f})', density=True)
ax2_twin = ax2.twinx()
ax2_twin.hist(amount_scaled, bins=50, alpha=0.5, color='#1f77b4',
              label=f'Scaled (mean≈0)', density=True)
ax2.set_xlabel('Amount')
ax2.set_ylabel('Density (original)', color='#ff7f0e')
ax2_twin.set_ylabel('Density (scaled)', color='#1f77b4')
ax2.set_title('StandardScaler: Amount Before vs After\n'
              'Mean shifted to 0, std to 1', fontsize=10)
ax2.legend(loc='upper right')
ax2_twin.legend(loc='center right')

plt.suptitle('Pipeline Inspection: What the Scaler Learned', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Pipeline + Cross-Validation: Leakage-Free Evaluation

**Why this is critical:** When you call `cross_val_score(pipeline, X, y, cv=5)`, sklearn's cross-validation engine calls `pipeline.fit(X_train_fold, y_train_fold)` for each fold. This means:

- The scaler is fit on the *training portion of each fold* — not on the full dataset
- The encoder is fit on the *training portion of each fold* — not on the full dataset
- Each validation fold is transformed using parameters from that fold's training data

This is **automatically leakage-free** because the Pipeline enforces fit-on-train/transform-on-validation.

**In contrast:** `cross_val_score(model, X_preprocessed_on_full_data, y, cv=5)` is *leaky* because `X_preprocessed_on_full_data` was already created using information from all validation folds.


In [ ]:
# ============================================================
# CELL 9: Pipeline + cross_val_score — leakage-free evaluation
# ============================================================

# StratifiedKFold preserves the class ratio (4% fraud) in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ---- CORRECT: cross-validate the full Pipeline ----
# The pipeline is freshly fit on each training fold — no leakage
cv_scores_pipeline = cross_val_score(
    full_pipeline,    # the whole pipeline, not just the model
    X_all,            # full feature matrix (no preprocessing done yet)
    y_all,            # full target vector
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1         # use all CPU cores for speed
)

# ---- INCORRECT (for comparison): pre-process all data, then cross-validate model ----
# This is leaky: the scaler was fit on full data including validation folds
# We'll simulate it by running cross_val_score on a model with already-preprocessed data
# (Note: to keep this fair, we'll use the manual preprocessing from Cell 4)
X_all_manual = np.hstack([  # preprocessing on full data = leaky
    scaler.fit_transform(num_imputer.fit_transform(X_all[NUM_COLS])),
    ohe.fit_transform(cat_imputer.fit_transform(X_all[CAT_COLS]))
])

# Now cross-validate just the model on the already-preprocessed (leaky) data
cv_scores_leaky = cross_val_score(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    X_all_manual,   # preprocessed on FULL data before CV — leaky
    y_all,
    cv=cv,
    scoring='roc_auc'
)

print('Cross-Validation Results (5-fold Stratified)')
print('=' * 55)
print(f'Pipeline CV (leakage-free):')
print(f'  Per-fold AUC: {np.round(cv_scores_pipeline, 4)}')
print(f'  Mean ± Std:   {cv_scores_pipeline.mean():.4f} ± {cv_scores_pipeline.std():.4f}')
print()
print(f'Pre-processed CV (leaky):')
print(f'  Per-fold AUC: {np.round(cv_scores_leaky, 4)}')
print(f'  Mean ± Std:   {cv_scores_leaky.mean():.4f} ± {cv_scores_leaky.std():.4f}')
print()
print('The leaky version tends to report a slightly inflated AUC.')
print('The Pipeline version is the honest estimate of production performance.')

In [ ]:
# ============================================================
# CELL 10: Visualise Pipeline CV vs Leaky CV results
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- Left: Per-fold AUC comparison ----
ax = axes[0]
folds = np.arange(1, 6)
width = 0.35
ax.bar(folds - width/2, cv_scores_pipeline, width,
       label='Pipeline CV (correct)', color='#2ca02c', alpha=0.85)
ax.bar(folds + width/2, cv_scores_leaky, width,
       label='Pre-processed CV (leaky)', color='#d62728', alpha=0.85)
ax.set_xlabel('Fold Number')
ax.set_ylabel('ROC-AUC')
ax.set_xticks(folds)
ax.set_ylim(0.5, 1.0)
ax.set_title('Per-Fold AUC: Pipeline vs Leaky Approach\n'
             'Leaky approach systematically inflates AUC', fontsize=10)
ax.legend()

# ---- Right: Box plot comparison ----
ax2 = axes[1]
bp = ax2.boxplot(
    [cv_scores_pipeline, cv_scores_leaky],
    labels=['Pipeline\n(Correct)', 'Pre-processed\n(Leaky)'],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
# Colour boxes
bp['boxes'][0].set_facecolor('#2ca02c')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#d62728')
bp['boxes'][1].set_alpha(0.7)
ax2.set_ylabel('ROC-AUC Across Folds')
ax2.set_title('AUC Distribution: 5-Fold CV\nPipeline gives honest variance estimate', fontsize=10)
ax2.set_ylim(0.5, 1.0)

# Annotate with mean AUC
for i, (scores, color) in enumerate([(cv_scores_pipeline, '#2ca02c'), (cv_scores_leaky, '#d62728')], start=1):
    ax2.text(i, scores.mean() + 0.01, f'μ={scores.mean():.4f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold', color=color)

plt.suptitle('Pipeline CV vs Pre-processed CV: Leakage in Evaluation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## GridSearchCV with Pipelines: Tuning Preprocessing AND the Model

One of the most powerful features of Pipelines is that `GridSearchCV` can tune **both** preprocessing hyperparameters and model hyperparameters simultaneously — and it does so correctly (no leakage in any fold).

**Parameter naming convention:** `step_name__parameter_name`
- To tune the `C` parameter in `LogisticRegression` under step name `'model'`: `model__C`
- To tune the `n_neighbors` parameter in `KNNImputer` under step `'preprocessor'` → `'num'` → `'imputer'`: `preprocessor__num__imputer__n_neighbors`

The double underscore `__` is the path separator through nested steps.


In [ ]:
# ============================================================
# CELL 11: Build a Pipeline with KNNImputer for GridSearchCV
# ============================================================

# KNNImputer fills missing values using k nearest neighbours
# n_neighbors is a hyperparameter we want to tune
numeric_pipeline_knn = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),  # n_neighbors will be grid-searched
    ('scaler',  StandardScaler())
])

categorical_pipeline_knn = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor_knn = ColumnTransformer(transformers=[
    ('num', numeric_pipeline_knn, NUM_COLS),
    ('cat', categorical_pipeline_knn, CAT_COLS),
])

pipeline_knn = Pipeline(steps=[
    ('preprocessor', preprocessor_knn),
    ('model', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ))
])

print('Pipeline with KNNImputer created.')
print()
print('Parameter grid will search over:')
print('  - preprocessor__num__imputer__n_neighbors: [3, 5, 7]')
print('  - model__C: [0.01, 0.1, 1.0, 10.0]')
print()
print('Note the naming convention: step_name__substep_name__parameter')

In [ ]:
# ============================================================
# CELL 12: GridSearchCV over Pipeline hyperparameters
# ============================================================

# Parameter grid: we search preprocessing AND model hyperparameters
param_grid = {
    # Tune the KNNImputer (inside preprocessor → num sub-pipeline)
    'preprocessor__num__imputer__n_neighbors': [3, 5, 7],
    # Tune the regularisation strength C in LogisticRegression
    # Small C = strong regularisation (simpler model)
    # Large C = weak regularisation (more complex model)
    'model__C': [0.01, 0.1, 1.0, 10.0]
}

# GridSearchCV will try all 3 × 4 = 12 combinations
# Each combination is evaluated with 3-fold stratified CV
# Total fits = 12 combinations × 3 folds = 36 Pipeline fits
grid_search = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,          # run in parallel
    verbose=0,          # set to 1 for progress output
    refit=True          # refit the best model on the full training set
)

# Fit on training data only — test data is never touched
grid_search.fit(X_train, y_train)

# ---- Results ----
print('GridSearchCV Results')
print('=' * 45)
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV AUC:     {grid_search.best_score_:.4f}')
print()

# Evaluate best Pipeline on held-out test set
best_pipeline = grid_search.best_estimator_
best_proba = best_pipeline.predict_proba(X_test)[:, 1]
best_auc   = roc_auc_score(y_test, best_proba)
print(f'Best Pipeline test AUC: {best_auc:.4f}')

# ---- Show results table ----
results_df = pd.DataFrame(grid_search.cv_results_)
relevant_cols = [
    'param_preprocessor__num__imputer__n_neighbors',
    'param_model__C',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]
print()
print('All parameter combinations (sorted by rank):')
print(
    results_df[relevant_cols]
    .rename(columns={
        'param_preprocessor__num__imputer__n_neighbors': 'knn_n',
        'param_model__C': 'C',
        'mean_test_score': 'mean_AUC',
        'std_test_score': 'std_AUC',
        'rank_test_score': 'rank'
    })
    .sort_values('rank')
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# CELL 13: Visualise GridSearchCV results as a heatmap
# ============================================================

# Pivot the results to build a heatmap: rows = n_neighbors, cols = C
pivot_grid = results_df.pivot_table(
    index='param_preprocessor__num__imputer__n_neighbors',
    columns='param_model__C',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.heatmap(
    pivot_grid,
    annot=True,
    fmt='.4f',
    cmap='YlGn',
    ax=ax,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean CV AUC'}
)

ax.set_title('GridSearchCV: AUC Heatmap\n'
             'Rows = KNN Imputer n_neighbors | Columns = LogReg C parameter\n'
             'Both preprocessing AND model hyperparameters tuned jointly',
             fontsize=11)
ax.set_xlabel('LogisticRegression C (regularisation strength)', fontsize=10)
ax.set_ylabel('KNNImputer n_neighbors', fontsize=10)

# Highlight the best cell
best_k = grid_search.best_params_['preprocessor__num__imputer__n_neighbors']
best_C = grid_search.best_params_['model__C']
ax.set_title(
    ax.get_title() + f'\nBest: n_neighbors={best_k}, C={best_C}  →  AUC={grid_search.best_score_:.4f}',
    fontsize=10
)

plt.tight_layout()
plt.show()

---
## Saving and Loading Pipelines with joblib

**Why save the entire Pipeline?**

If you save *only* the model weights, you still need to remember (and correctly apply) every preprocessing step at inference time:
- "What was the imputation strategy?"
- "What was the scaler mean for `amount`?"
- "What categories did the OneHotEncoder see during training?"
- "What order were the columns in?"

Get any of these wrong and your production predictions are garbage — silently wrong.

When you save the entire Pipeline with `joblib`, **all learned parameters are saved together**: scaler means, imputer medians, encoder vocabulary, model weights — everything. At inference time, you load one object and call `.predict()`. Nothing to remember, nothing to get wrong.


In [ ]:
# ============================================================
# CELL 14: Save and reload the Pipeline with joblib
# ============================================================

# ---- Save the best Pipeline to disk ----
# We use a temp directory so this notebook works anywhere without hard-coded paths
save_dir = tempfile.mkdtemp()  # creates a temporary directory
pipeline_path = os.path.join(save_dir, 'fraud_detection_pipeline.pkl')

# joblib.dump serialises the entire Python object (including all fitted params)
# compress=3 uses gzip compression to reduce file size
joblib.dump(best_pipeline, pipeline_path, compress=3)

# Check file size — the saved pipeline is a compact binary file
file_size_kb = os.path.getsize(pipeline_path) / 1024
print(f'Pipeline saved to: {pipeline_path}')
print(f'File size:         {file_size_kb:.1f} KB')
print()

# ---- Reload the Pipeline from disk ----
loaded_pipeline = joblib.load(pipeline_path)
print(f'Loaded pipeline type: {type(loaded_pipeline).__name__}')
print()

# ---- Verify: predictions must be IDENTICAL ----
# If the reload is correct, every prediction should exactly match the original
original_proba = best_pipeline.predict_proba(X_test)[:, 1]
loaded_proba   = loaded_pipeline.predict_proba(X_test)[:, 1]

predictions_match = np.allclose(original_proba, loaded_proba, atol=1e-10)
max_diff = np.abs(original_proba - loaded_proba).max()

print(f'Predictions identical: {predictions_match}')
print(f'Maximum difference:    {max_diff:.2e}')
print()
print('Original AUC:     ', round(roc_auc_score(y_test, original_proba), 6))
print('Loaded Pipeline AUC:', round(roc_auc_score(y_test, loaded_proba), 6))
print()
print('The saved Pipeline includes: scaler means, imputer medians,')
print('encoder categories, AND model weights — all in one file.')

In [ ]:
# ============================================================
# CELL 15: Simulate production inference with the loaded Pipeline
# ============================================================

# At production time, new transactions arrive one or more at a time.
# The loaded pipeline handles them identically to training — no manual preprocessing.

# Simulate 5 new incoming transactions with some missing values
new_transactions = pd.DataFrame({
    'amount':                [1250.00, 42.50, np.nan, 8900.00, 15.00],
    'hour_of_day':           [2.0, 14.0, 8.0, np.nan, 19.0],
    'distance_from_home_km': [850.0, np.nan, 3.2, 12000.0, 1.5],
    'n_prev_transactions':   [0.0, 3.0, np.nan, 1.0, 5.0],
    'merchant_category':     ['online', 'grocery', np.nan, 'travel', 'restaurant'],
    'card_type':             ['visa', np.nan, 'mastercard', 'amex', 'discover']
})

print('New transactions to score:')
print(new_transactions.to_string())
print()

# The Pipeline handles imputation, scaling, and encoding automatically
fraud_probabilities = loaded_pipeline.predict_proba(new_transactions)[:, 1]
fraud_predictions   = loaded_pipeline.predict(new_transactions)

results = pd.DataFrame({
    'transaction_id':    range(1, 6),
    'amount':            new_transactions['amount'],
    'merchant_category': new_transactions['merchant_category'],
    'fraud_probability': fraud_probabilities.round(4),
    'prediction':        ['FRAUD' if p == 1 else 'LEGITIMATE' for p in fraud_predictions]
})

print('Fraud Scoring Results:')
print('=' * 65)
print(results.to_string(index=False))
print()
print('Missing values were automatically imputed by the Pipeline.')
print('No manual preprocessing was needed at inference time.')

In [ ]:
# ============================================================
# CELL 16: Final comparison — all approaches side by side
# ============================================================

# Collect all results from this notebook
results_summary = pd.DataFrame([
    {'Approach':         'Manual (7 separate steps)',
     'AUC':              manual_auc,
     'Leakage Risk':    'High',
     'CV Ease':         'Manual loop required',
     'Deployment':      'Save 4+ separate objects'},

    {'Approach':         'Pipeline (basic)',
     'AUC':              pipe_auc,
     'Leakage Risk':    'None',
     'CV Ease':         'cross_val_score(pipeline, ...)',
     'Deployment':      'One joblib.dump()'},

    {'Approach':         'Pipeline + GridSearchCV',
     'AUC':              best_auc,
     'Leakage Risk':    'None',
     'CV Ease':         'GridSearchCV(pipeline, ...)',
     'Deployment':      'One joblib.dump()'},

    {'Approach':         'Loaded Pipeline (production)',
     'AUC':              roc_auc_score(y_test, loaded_proba),
     'Leakage Risk':    'None',
     'CV Ease':         'N/A — inference only',
     'Deployment':      'joblib.load() → predict()'},
])

print('Summary: All Approaches Compared')
print('=' * 80)
print(results_summary.to_string(index=False))

# ---- AUC bar chart ----
fig, ax = plt.subplots(figsize=(10, 5))
approach_names = ['Manual\n(7 steps)', 'Pipeline\n(basic)',
                  'Pipeline +\nGridSearchCV', 'Loaded\n(production)']
auc_vals = [manual_auc, pipe_auc, best_auc, roc_auc_score(y_test, loaded_proba)]
colors = ['#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

bars = ax.bar(approach_names, auc_vals, color=colors, alpha=0.85, edgecolor='white')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('ROC-AUC on Test Set', fontsize=11)
ax.set_title('AUC Comparison: Manual vs Pipeline Approaches\n'
             'Pipeline equals or beats manual while eliminating leakage risk',
             fontsize=12)

for bar, val in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

---
## Pipeline Quick Reference Card

```python
# ---- Basic Pipeline ----
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LogisticRegression())
])

# ---- ColumnTransformer ----
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, cat_cols)
])

# ---- Train ----
pipe.fit(X_train, y_train)

# ---- Predict ----
pipe.predict(X_test)
pipe.predict_proba(X_test)

# ---- Cross-validate (leakage-free) ----
cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')

# ---- GridSearch (preprocessing + model) ----
GridSearchCV(pipe, {'model__C': [0.1, 1, 10]}, cv=5)

# ---- Inspect fitted steps ----
pipe.named_steps['scaler'].mean_
pipe[:-1].transform(X)          # get preprocessed features
pipe.named_steps['model'].coef_ # model weights

# ---- Save / Load ----
joblib.dump(pipe, 'model.pkl')
pipe = joblib.load('model.pkl')
```

---
## Self-Check Questions

---

**Question 1:** In a Pipeline with steps `[Imputer, Scaler, Model]`, when you call `pipeline.fit(X_train, y_train)`, what happens internally for each step?

<details>
<summary>Show Answer</summary>

Internally, sklearn iterates through the steps and:

1. **Imputer:** Calls `imputer.fit_transform(X_train, y_train)` — the imputer *learns* the median (or chosen strategy) from `X_train` and replaces NaNs. The output `X_imputed` is passed forward.
2. **Scaler:** Calls `scaler.fit_transform(X_imputed, y_train)` — the scaler *learns* the mean and std from `X_imputed` (derived purely from training data) and standardises the values. The output `X_scaled` is passed forward.
3. **Model:** Calls `model.fit(X_scaled, y_train)` — the model *learns* its weights from the scaled, imputed training data.

**Key:** All three steps learn their parameters exclusively from `X_train`. No test data is involved at any stage.

</details>

---

**Question 2:** Why is `cross_val_score(pipeline, X, y)` leakage-free, but `cross_val_score(model, X_preprocessed, y)` might not be?

<details>
<summary>Show Answer</summary>

When `cross_val_score` receives a **Pipeline**, it calls `pipeline.fit(X_train_fold, y_train_fold)` for each fold. This causes every preprocessing step (imputer, scaler, encoder) to be fitted on *only the training portion of each fold*. The validation fold is then transformed using those fold-specific parameters — it never influences the fitting.

When `cross_val_score` receives just a **model** with `X_preprocessed` already computed, `X_preprocessed` was created by fitting preprocessing on the *full dataset* (including all validation folds). The scaler's mean and the imputer's median were computed using validation fold data. This is leakage — the validation set's statistics influenced how the data was prepared before it was evaluated.

**Rule:** Cross-validate the entire Pipeline, not just the final model.

</details>

---

**Question 3:** You trained a Pipeline including a `OneHotEncoder`. At production, a new merchant category appears (`'crypto_exchange'`) that wasn't in the training data. What happens and how do you handle it?

<details>
<summary>Show Answer</summary>

Without the `handle_unknown` parameter, `OneHotEncoder` will raise a `ValueError` when it encounters an unknown category.

**Fix:** Set `handle_unknown='ignore'` when building the encoder:
```python
OneHotEncoder(drop='first', handle_unknown='ignore')
```

With `handle_unknown='ignore'`, the encoder represents any unknown category as an all-zeros row (i.e., "none of the known categories"). This is a reasonable default: the model makes a prediction based on the other features without any category-specific signal.

Note that we already included `handle_unknown='ignore'` in our Pipeline definition in this notebook — this is a best practice for production fraud detection models where new merchant types appear regularly.

</details>

---

**Question 4:** What is the advantage of saving the entire Pipeline with joblib rather than just saving the model weights?

<details>
<summary>Show Answer</summary>

Saving just the model weights requires you to correctly reconstruct and apply every preprocessing step at inference time manually. This creates multiple failure points:

1. **Scaler mismatch:** If you forget the exact mean and std used during training, you'll scale inputs incorrectly and get wrong predictions with no error message.
2. **Imputation mismatch:** If you impute with a different value than training (e.g., current median instead of training median), feature distributions shift.
3. **Encoder inconsistency:** OneHotEncoder must use exactly the same category vocabulary and drop strategy as during training. A different column order will silently corrupt all predictions.
4. **Column order:** The model expects features in a specific order. Any reordering silently corrupts predictions.

Saving the entire Pipeline with `joblib.dump()` serialises **all learned parameters** in a single file. Loading with `joblib.load()` gives back an object where `.predict()` automatically applies the correct, training-time preprocessing — with no manual steps required. This is both safer and far simpler to maintain in production.

</details>


In [ ]:
# ============================================================
# CELL 17: Bonus — what happens with an unknown category at production
# ============================================================

# This demonstrates the handle_unknown='ignore' behaviour concretely

# Create a transaction with a category never seen during training
unknown_category_txn = pd.DataFrame({
    'amount':                [500.0],
    'hour_of_day':           [3.0],
    'distance_from_home_km': [1200.0],
    'n_prev_transactions':   [0.0],
    'merchant_category':     ['crypto_exchange'],  # NOT in training data
    'card_type':             ['visa']
})

print('Transaction with unknown merchant category ("crypto_exchange"):')
print(unknown_category_txn.to_string())
print()

# The loaded pipeline uses handle_unknown='ignore', so this will NOT raise an error
try:
    prob = loaded_pipeline.predict_proba(unknown_category_txn)[:, 1][0]
    pred = 'FRAUD' if loaded_pipeline.predict(unknown_category_txn)[0] == 1 else 'LEGITIMATE'
    print(f'Fraud probability: {prob:.4f}')
    print(f'Prediction:        {pred}')
    print()
    print('handle_unknown="ignore" encoded the unknown category as all zeros.')
    print('The model made a prediction using only the numeric features.')
    print('No error was raised — safe for production use.')
except Exception as e:
    print(f'ERROR: {e}')
    print('This would happen without handle_unknown="ignore".')

print()
print('Best practice: Log unknown categories in production to monitor')
print('data drift — if new categories appear frequently, retrain the model.')